In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import random
import os
from tqdm import tqdm # Biblioteca para barra de progresso

In [2]:
def plot_beat_segment(segment_df, title="Segmento de Batimento Cardíaco"):
    """Função auxiliar para plotar um segmento de batimento de um DataFrame."""
    if segment_df is None or segment_df.empty:
        print("DataFrame vazio. Nada para plotar.")
        return

    plt.figure(figsize=(12, 6))
    plt.plot(segment_df["sample #"], segment_df["amplitude"], label="Sinal")
    plt.title(title)
    plt.xlabel("Número da Amostra")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

In [3]:
def save_augmented(segment_df, save_path, aug_name):
    """Salva segmento em CSV no formato padrão."""
    augmented_df = segment_df.copy()
    augmented_df["channel_0"] = augmented_df["amplitude"]
    augmented_df = augmented_df[["channel_0", "sample #", "type"]]
    augmented_df.to_csv(save_path, index=False)

In [4]:
# Versão ajustada da augment_jitter
def augment_jitter(segment_df, sigma_factor=0.02, variable_sigma=True, seed=None):
    # Removi save_path e number dos argumentos
    rng = np.random.default_rng(seed)
    augmented_df = segment_df.copy()
    base_sigma = sigma_factor * np.std(segment_df["amplitude"])
    
    if variable_sigma:
        sigma_series = base_sigma * rng.lognormal(mean=0, sigma=0.25, size=len(segment_df))
    else:
        sigma_series = np.full(len(segment_df), base_sigma)
        
    noise = rng.normal(0, sigma_series)
    augmented_df["amplitude"] += noise
    
    # A linha que salvava o arquivo foi removida
    return augmented_df

In [5]:
# Coloque esta função no lugar da sua versão antiga
def get_beat_interval_robust_optimized(df, all_peaks, target_rows, nth=0, channel="channel_0"):
    """
    Versão otimizada que recebe um DataFrame e picos pré-calculados.
    """
    # Não precisa mais ler o CSV nem encontrar os picos aqui
    if nth >= len(target_rows):
        print(f"Aviso: nth={nth} está fora do alcance. Anotações encontradas: {len(target_rows)}")
        return None

    center_sample = int(target_rows.iloc[nth]["sample #"])

    # Encontra o índice do pico mais próximo da anotação
    center_peak_index = np.argmin(np.abs(all_peaks - center_sample))
    
    # Verificação de borda
    if center_peak_index == 0 or center_peak_index >= len(all_peaks) - 1:
        return None

    # Pega os picos vizinhos
    start_peak = all_peaks[center_peak_index - 1]
    end_peak = all_peaks[center_peak_index + 1]
    
    # Extrai o segmento
    mask = (df["sample #"] >= start_peak) & (df["sample #"] <= end_peak)
    segment_df = df.loc[mask].copy()

    # Renomeia a coluna para o padrão "amplitude"
    if channel in segment_df.columns:
        segment_df.rename(columns={channel: "amplitude"}, inplace=True)
    
    return segment_df

In [7]:
# --- Execução Otimizada ---
if __name__ == '__main__':
    csv_file = "mitbih_all_records_renumerada.csv"
    output_file = "augmented_beats_L_jitter.csv" # Um único arquivo de saída
    target_type = "R"
    num_augmentations = 7200
    
    # =================================================================
    # PASSO 1: LER E PRÉ-CALCULAR TUDO FORA DO LOOP
    # =================================================================
    print("1/4 - Carregando o arquivo CSV principal (apenas uma vez)...")
    df_main = pd.read_csv(csv_file)
    
    print("2/4 - Encontrando todas as anotações do tipo '{}'...".format(target_type))
    target_rows = df_main[df_main["type"] == target_type].reset_index(drop=True)
    
    if len(target_rows) < num_augmentations:
        print(f"Aviso: Pedido de {num_augmentations} aumentos, mas apenas {len(target_rows)} batimentos do tipo '{target_type}' foram encontrados.")
        num_augmentations = len(target_rows)

    print("3/4 - Detectando todos os picos R principais (apenas uma vez)...")
    # Use os mesmos parâmetros de altura/distância que funcionaram para você
    peaks, _ = find_peaks(df_main["channel_0"].values, distance=180, height=0.25)
    
    # =================================================================
    # PASSO 2: EXECUTAR O LOOP EM MEMÓRIA
    # =================================================================
    print(f"4/4 - Gerando {num_augmentations} aumentos de dados...")
    
    all_augmented_segments = [] # Lista para guardar os resultados
    
    # Usando tqdm para uma barra de progresso
    for i in tqdm(range(num_augmentations), desc="Processando Batimentos"):
        # Pega o segmento usando a função otimizada
        segment_df = get_beat_interval_robust_optimized(df_main, peaks, target_rows, nth=i)
        
        if segment_df is not None:
            # Aplica o Jitter (esta função já é rápida)
            # A função augment_jitter precisa ser levemente ajustada para não salvar o arquivo
            augmented_segment = augment_jitter(segment_df, sigma_factor=0.02) # Removi o save_path daqui
            
            # Adiciona um ID único para cada batimento aumentado
            augmented_segment['beat_id'] = i 
            
            all_augmented_segments.append(augmented_segment)
            
    # =================================================================
    # PASSO 3: SALVAR TUDO DE UMA VEZ
    # =================================================================
    if all_augmented_segments:
        print(f"Consolidando e salvando {len(all_augmented_segments)} segmentos em um único arquivo...")
        final_df = pd.concat(all_augmented_segments, ignore_index=True)
        final_df.to_csv(output_file, index=False)
        print(f"Processo concluído! Arquivo salvo em: {output_file}")
    else:
        print("Nenhum segmento foi gerado.")

1/4 - Carregando o arquivo CSV principal (apenas uma vez)...
2/4 - Encontrando todas as anotações do tipo 'L'...
3/4 - Detectando todos os picos R principais (apenas uma vez)...
4/4 - Gerando 8000 aumentos de dados...


Processando Batimentos: 100%|███████████████████████████████████████████████████████| 8000/8000 [10:31<00:00, 12.67it/s]


Consolidando e salvando 8000 segmentos em um único arquivo...
Processo concluído! Arquivo salvo em: augmented_beats_L_jitter.csv
